# Composition assignment

This notebook assigns potential **elemental compositions** to
unmatched peaks in a sample spectrum.

### User input

The following cells require user configuration:

| Cell              | What to set                                                                          |
| ----------------- | ------------------------------------------------------------------------------------ |
| **Select sample** | `dataset`, `batch` to list samples; `sample_name` to pick one                        |
| **Parameters**    | `ionization_mechanisms`: list of mechanism names (e.g. `["-H+", "+^NO3-"]`)          |
|                   | `mz_tolerance_ppm`: m/z tolerance for candidate lookup                               |
|                   | `formula_ranges`: allowed elements and counts (e.g. `"C0-100 H0-100 O0-100 N0-100"`) |
|                   | `max_candidates`: max formulas to evaluate per peak                                  |
|                   | `min_match_score`: threshold to accept a candidate (0-1)                             |
|                   | `max_peaks`: limit number of peaks to process (`None` = all)                         |

### Workflow

1. Load peaks for the selected sample; split into matched and unmatched
2. For each unmatched peak (ascending m/z):
   - Query candidate formulas via `cheminfo.query_by_mz`
   - Score each candidate against the sample's isotope pattern via `matching.match_compounds`
   - Pick the best candidate by `match_score`
   - Assign **all isotope peaks** of the winning ion, subsequent peaks that
     belong to the same isotope pattern are skipped automatically
3. Collect results and visualise: annotated mass spectrum, m/z error plot


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

from mascope_sdk import MascopeClient


pio.templates.default = "plotly_dark"  # or "plotly_white"

mascope = MascopeClient(workspace="My Workspace")

### Select sample and load peaks


In [ ]:
# List samples in a batch to pick one
samples = mascope.samples.list(dataset="My Dataset", batch="My Batch")
samples[["sample_item_id", "sample_item_name", "datetime_utc"]]

In [ ]:
# Pick a sample (by name or index)
sample_name = "Sample 1"  # <- Change this to the name of your sample
sample_row = samples[samples["sample_item_name"] == sample_name].iloc[0]
sample_id = sample_row["sample_item_id"]

print(f"Sample: {sample_name}")
print(f"ID:     {sample_id}")

In [ ]:
# Load peaks with matches
peaks = mascope.samples.get_peaks(sample_id)

# Split into matched and unmatched
matched_peaks = peaks[peaks["target_compound_formula"].notna()]
unmatched_peaks = peaks[peaks["target_compound_formula"].isna()].copy()

print(f"Total peaks: {peaks['peak_id'].nunique()}")
print(f"  Matched:   {matched_peaks['peak_id'].nunique()}")
print(f"  Unmatched: {unmatched_peaks['peak_id'].nunique()}")
unmatched_peaks[["peak_id", "mz", "height"]].sort_values("mz")

### Parameters


In [ ]:
# --- Composition assignment parameters ---

# Ionization mechanisms to consider (human-readable names, e.g. "-H+", "+^NO3-")
ionization_mechanisms = ["-H+", "+Br-"]

# m/z tolerance for candidate formula lookup (ppm)
mz_tolerance_ppm = 3.0

# Element ranges for formula generation
formula_ranges = "C0-100 H0-100 O0-100 N0-100"

# Maximum number of candidate formulas per peak
max_candidates = 10

# Minimum match score to accept a candidate
min_match_score = 0.9

# Maximum number of peaks to process (set None for all)
max_peaks = 50

### Resolve ionization mechanisms

Resolve the human-readable ionization mechanism names to their IDs.


In [ ]:
# Load all mechanisms and resolve names to IDs
all_mechanisms = mascope.ionization.list()
name_to_id = dict(
    zip(
        all_mechanisms["ionization_mechanism"],
        all_mechanisms["ionization_mechanism_id"],
    )
)

mechanism_ids = []
for name in ionization_mechanisms:
    if name in name_to_id:
        mechanism_ids.append(name_to_id[name])
    else:
        available = sorted(name_to_id.keys())
        raise ValueError(
            f"Ionization mechanism '{name}' not found.\nAvailable: {available}"
        )

print(f"Resolved {len(mechanism_ids)} ionization mechanism(s):")
for name, mid in zip(ionization_mechanisms, mechanism_ids):
    print(f"  {name} → {mid}")

### Assign compositions

Peaks are processed in **ascending m/z** order. For each unmatched peak:

1. Query candidate formulas using `cheminfo.query_by_mz`
2. For each candidate, run `matching.match_compounds` to score the
   isotope pattern match against the sample
3. Pick the best candidate by `match_score`
4. Assign **all isotope peaks** of the best ion (not just the trigger peak),
   removing them from the queue so they are not processed again


In [ ]:
# Sort unmatched peaks by m/z (ascending)
to_assign = unmatched_peaks.sort_values("mz").drop_duplicates(subset=["peak_id"])
if max_peaks is not None:
    to_assign = to_assign.head(max_peaks)

# Build a peak_id → row lookup for resolving isotope siblings
peak_lookup = to_assign.set_index("peak_id")

# Track which peaks have already been assigned (by an earlier ion's isotopes)
assigned_ids = set()

print(f"Processing {len(to_assign)} unmatched peaks ...")

results = []
all_candidates = []  # every scored candidate (not just the winner)
n_skipped = 0

for i, (_, peak) in enumerate(to_assign.iterrows()):
    peak_id = peak["peak_id"]
    peak_mz = peak["mz"]
    peak_height = peak["height"]

    # Skip if already assigned as an isotope sibling of a previous peak
    if peak_id in assigned_ids:
        n_skipped += 1
        continue

    if (i + 1) % 25 == 0 or i == 0:
        print(f"  [{i + 1}/{len(to_assign)}] m/z = {peak_mz:.4f}")

    # Step 1: Query candidate formulas
    candidates = mascope.cheminfo.query_by_mz(
        mz=peak_mz,
        ionization_mechanism_ids=mechanism_ids,
        formula_ranges=formula_ranges,
        mz_tolerance=mz_tolerance_ppm,
        limit=max_candidates,
    )

    if not candidates:
        results.append(
            {
                "peak_id": peak_id,
                "peak_mz": peak_mz,
                "peak_height": peak_height,
                "target_compound_formula": None,
                "ionization_mechanism": None,
                "target_ion_formula": None,
                "target_isotope_formula": None,
                "mz_error_ppm": None,
                "match_score": None,
                "dbe": None,
                "n_candidates": 0,
            }
        )
        continue

    # Step 2: Score each candidate via match_compounds
    best_score = -1.0
    best_ion = None
    best_compound_formula = None
    best_dbe = None

    # Deduplicate candidates by formula (same formula via different mechanisms)
    seen_formulas = set()
    unique_candidates = []
    for c in candidates:
        formula = c["target_compound_formula"]
        if formula not in seen_formulas:
            seen_formulas.add(formula)
            unique_candidates.append(c)

    for c in unique_candidates:
        formula = c["target_compound_formula"]

        match_data = mascope.matching.match_compounds(
            sample_id=sample_id,
            formulas=[formula],
        )

        if not match_data:
            continue

        compound = match_data[0]

        # Find the best ion for this compound
        ions = compound.get("children", [])
        if not ions:
            continue

        ion = max(ions, key=lambda x: x.get("match_score", 0))
        score = ion.get("match_score", 0.0)

        # Resolve mechanism name for this candidate
        cand_mechanism = None
        if ion.get("ionization_mechanism_id"):
            mech_row = all_mechanisms[
                all_mechanisms["ionization_mechanism_id"]
                == ion["ionization_mechanism_id"]
            ]
            if not mech_row.empty:
                cand_mechanism = mech_row.iloc[0]["ionization_mechanism"]

        # Collect this candidate (regardless of whether it wins)
        all_candidates.append(
            {
                "peak_id": peak_id,
                "peak_mz": peak_mz,
                "target_compound_formula": formula,
                "ionization_mechanism": cand_mechanism,
                "target_ion_formula": ion.get("target_ion_formula"),
                "match_score": score,
                "dbe": c["target_compound_unsaturation"],
            }
        )

        if score > best_score:
            best_score = score
            best_ion = ion
            best_compound_formula = formula
            best_dbe = c["target_compound_unsaturation"]

    # Step 3: Assign all isotope children of the best ion
    if best_ion is not None and best_score >= min_match_score:
        # Resolve mechanism name
        mechanism_name = None
        if best_ion.get("ionization_mechanism_id"):
            mech_row = all_mechanisms[
                all_mechanisms["ionization_mechanism_id"]
                == best_ion["ionization_mechanism_id"]
            ]
            if not mech_row.empty:
                mechanism_name = mech_row.iloc[0]["ionization_mechanism"]

        ion_formula = best_ion.get("target_ion_formula")

        for iso in best_ion.get("children", []):
            iso_score = iso.get("match_score", 0.0)
            if iso_score < min_match_score:
                continue
            iso_peak_id = iso.get("sample_peak_id")
            if iso_peak_id is None:
                continue

            # Look up this peak's m/z and height from our data
            if iso_peak_id in peak_lookup.index:
                iso_row = peak_lookup.loc[iso_peak_id]
                iso_mz = iso_row["mz"]
                iso_height = iso_row["height"]
            else:
                # Peak not in our unmatched set (may already be matched)
                continue

            assigned_ids.add(iso_peak_id)
            results.append(
                {
                    "peak_id": iso_peak_id,
                    "peak_mz": iso_mz,
                    "peak_height": iso_height,
                    "target_compound_formula": best_compound_formula,
                    "ionization_mechanism": mechanism_name,
                    "target_ion_formula": ion_formula,
                    "target_isotope_formula": iso.get("target_isotope_formula"),
                    "mz_error_ppm": iso.get("match_mz_error"),
                    "match_score": iso.get("match_score"),
                    "n_candidates": len(unique_candidates),
                    "dbe": best_dbe,
                }
            )
    else:
        results.append(
            {
                "peak_id": peak_id,
                "peak_mz": peak_mz,
                "peak_height": peak_height,
                "target_compound_formula": None,
                "ionization_mechanism": None,
                "target_ion_formula": None,
                "target_isotope_formula": None,
                "mz_error_ppm": None,
                "match_score": None,
                "n_candidates": len(unique_candidates) if candidates else 0,
                "dbe": None,
            }
        )

candidates_df = pd.DataFrame(all_candidates)

print(
    f"Done. {len(results)} rows from {len(to_assign)} peaks "
    f"({n_skipped} skipped as isotope siblings)."
)
print(f"Collected {len(candidates_df)} total candidates across all peaks.")

### Results


In [ ]:
assignments = pd.DataFrame(results)

assigned = assignments[assignments["target_compound_formula"].notna()]
unassigned = assignments[assignments["target_compound_formula"].isna()]

print(f"Assigned:   {len(assigned)} / {len(assignments)} peaks")
print(f"Unassigned: {len(unassigned)} / {len(assignments)} peaks")

assigned.sort_values("match_score", ascending=False)

In [ ]:
# All scored candidates (including runners-up) — for alternative selection criteria
# Each row is one (peak, candidate formula) pair with its match score.
# The "rank" column shows how each candidate ranks within its peak (1 = best).
candidates_df["rank"] = (
    candidates_df.groupby("peak_id")["match_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# Show peaks with multiple candidates
multi = candidates_df[
    candidates_df["peak_id"].isin(
        candidates_df.groupby("peak_id").filter(lambda g: len(g) > 1)["peak_id"]
    )
]
print(f"{multi['peak_id'].nunique()} peaks have multiple candidates")
multi.sort_values(["peak_mz", "rank"])

In [ ]:
import numpy as np

# Annotated mass spectrum: original matched, newly assigned, and unassigned
# Build a combined DataFrame for the plot
original_matched = matched_peaks.drop_duplicates(subset=["peak_id"]).assign(
    status="matched (existing)"
)[
    [
        "peak_id",
        "mz",
        "height",
        "status",
        "target_compound_formula",
        "target_ion_formula",
        "ionization_mechanism",
    ]
]
newly_assigned = assigned.rename(
    columns={"peak_mz": "mz", "peak_height": "height"}
).assign(status="matched (new)")[
    [
        "peak_id",
        "mz",
        "height",
        "status",
        "target_compound_formula",
        "target_ion_formula",
        "ionization_mechanism",
    ]
]
still_unknown = unassigned.rename(
    columns={"peak_mz": "mz", "peak_height": "height"}
).assign(status="unmatched")[
    [
        "peak_id",
        "mz",
        "height",
        "status",
        "target_compound_formula",
        "target_ion_formula",
        "ionization_mechanism",
    ]
]


df = pd.concat([original_matched, newly_assigned, still_unknown], ignore_index=True)

df = df.sort_values(["mz"])
df["mass_defect"] = df["mz"] - df["mz"].round()
df["log_intensity"] = np.log10(df["height"].clip(lower=1))

fig = px.scatter(
    df,
    x="mz",
    y="mass_defect",
    color="status",
    color_discrete_map={
        "matched (existing)": "#636EFA",
        "matched (new)": "#00CC96",
        "unmatched": "#EF553B",
    },
    size="log_intensity",
    size_max=15,
    title="Mass spectrum: composition assignment overview",
    hover_data=[
        "target_compound_formula",
        "target_ion_formula",
        "ionization_mechanism",
    ],
)
fig.update_layout(
    xaxis_title="m/z [Th]",
    yaxis_title="Mass defect [Th]",
    yaxis_type="linear",
)
fig.show()

In [ ]:
# m/z error vs peak m/z for assigned compositions
fig = px.scatter(
    assigned.dropna(subset=["mz_error_ppm"]),
    x="peak_mz",
    y="mz_error_ppm",
    color="match_score",
    hover_data=[
        "target_compound_formula",
        "target_ion_formula",
        "ionization_mechanism",
    ],
    title="m/z error vs peak m/z for assigned compositions",
)
fig.update_layout(
    xaxis_title="Peak m/z [Th]",
    yaxis_title="m/z error [ppm]",
)
fig.show()